In [2]:
%pip install -q torch transformers bitsandbytes accelerate datasets evaluate nltk rouge-score sacrebleu pandas numpy matplotlib seaborn scikit-learn streamlit tqdm sentencepiece protobuf

Note: you may need to restart the kernel to use updated packages.


# Notebook 1 — Persiapan Data

Memuat dan validasi dataset prompt dari penelitian Oprea & Bâra (2026).
**CATATAN: Semua data dalam workflow ini adalah SIMULASI dari hasil jurnal, bukan eksekusi asli.**


In [8]:
import pandas as pd
import numpy as np

print('=' * 70)
print('📚 WORKFLOW SIMULASI JURNAL OPREA & BÂRA (2026)')
print('=' * 70)
print()
print('ℹ️  Semua data dalam workflow ini menggunakan:')
print('  • Hasil simulasi dari paper asli')
print('  • 3 model: GPT-2, LLaMA-2-7B-Chat, Qwen1.5-1.8B-Chat')
print('  • 2 GPU: RTX4070 dan RTX4080 Laptop')
print('  • 50 prompts untuk evaluasi text generation')
print('  • 15 prompts untuk evaluasi code generation')
print()
print('✅ Library berhasil dimuat')


📚 WORKFLOW SIMULASI JURNAL OPREA & BÂRA (2026)

ℹ️  Semua data dalam workflow ini menggunakan:
  • Hasil simulasi dari paper asli
  • 3 model: GPT-2, LLaMA-2-7B-Chat, Qwen1.5-1.8B-Chat
  • 2 GPU: RTX4070 dan RTX4080 Laptop
  • 50 prompts untuk evaluasi text generation
  • 15 prompts untuk evaluasi code generation

✅ Library berhasil dimuat


## 📚 Journal Source & Data Verification

**Paper Source:** Oprea, S.-V., & Bâra, A. (2026). "Quantized Transformers in Practice: Benchmarking Full- and Low-Precision LLMs across Two Processors." *Computers, Materials & Continua*, 87(3), 91. https://doi.org/10.32604/cmc.2026.078985

**Data Tables Referenced:**
- **Table 3**: Timing metrics and tokens/second for each model-GPU-precision configuration
- **Table 9**: Aggregate statistics (mean ± std) across 50 prompts
- **Table 13**: Code generation metrics (15 prompts)

**All data in this workflow is sourced directly from the journal—NO bias, NO modifications.**


## 1. Data simulasi dari Notebook 02 (Benchmark Results)


In [9]:
# Data benchmark dari Notebook 02 (Simulasi dari Table 3 jurnal)
benchmark_data = {
    'gpt2': {
        'RTX4070': {'FP16': (23.21, 11.03), 'INT8': (17.37, 14.74)},
        'RTX4080': {'FP16': (1.10, 231.73), 'INT8': (2.01, 127.36)}
    },
    'llama2': {
        'RTX4070': {'FP16': (1244.29, 0.21), 'INT8': (18.63, 13.74)},
        'RTX4080': {'FP16': (50.04, 5.12), 'INT8': (26.64, 9.61)}
    },
    'qwen': {
        'RTX4070': {'FP16': (12.66, 20.22), 'INT8': (11.78, 21.73)},
        'RTX4080': {'FP16': (13.61, 18.81), 'INT8': (11.08, 23.10)}
    }
}

print('📊 Data Benchmark (dari Notebook 02):')
print(f'  • Total model: 3 (GPT-2, LLaMA-2-7B, Qwen1.5-1.8B)')
print(f'  • Total GPU: 2 (RTX4070, RTX4080)')
print(f'  • Total konfigurasi: 12 (3 model × 2 GPU × 2 precision)')
print()
print('Informasi konfigurasi:')
for model in benchmark_data:
    for gpu in benchmark_data[model]:
        print(f'  ✓ {model.upper()}-{gpu}')


📊 Data Benchmark (dari Notebook 02):
  • Total model: 3 (GPT-2, LLaMA-2-7B, Qwen1.5-1.8B)
  • Total GPU: 2 (RTX4070, RTX4080)
  • Total konfigurasi: 12 (3 model × 2 GPU × 2 precision)

Informasi konfigurasi:
  ✓ GPT2-RTX4070
  ✓ GPT2-RTX4080
  ✓ LLAMA2-RTX4070
  ✓ LLAMA2-RTX4080
  ✓ QWEN-RTX4070
  ✓ QWEN-RTX4080


## 2. Dataset evaluasi metrik (dari Notebook 03)


In [ ]:
# Metrics dari Table 9 jurnal (Aggregate statistics 50 prompts - INT8)
metrics_data = [
    {'model': 'LLaMA-2-7B-Chat', 'gpu': 'RTX4070', 'bleu': 0.180, 'rouge1': 0.509, 'rougeL': 0.343},
    {'model': 'LLaMA-2-7B-Chat', 'gpu': 'RTX4080', 'bleu': 0.117, 'rouge1': 0.522, 'rougeL': 0.409},
    {'model': 'Qwen1.5-1.8B-Chat', 'gpu': 'RTX4070', 'bleu': 0.134, 'rouge1': 0.618, 'rougeL': 0.291},
    {'model': 'Qwen1.5-1.8B-Chat', 'gpu': 'RTX4080', 'bleu': 0.113, 'rouge1': 0.387, 'rougeL': 0.294},
]

df_metrics = pd.DataFrame(metrics_data)
print('📋 Tabel Metrik Kualitas (INT8, mean dari 50 prompts):')
print(df_metrics.to_string(index=False))
print()
print(f'✓ Total konfigurasi evaluasi: {len(df_metrics)}')


  kategori      tipe  jumlah
0     Kode      Kode      10
1     Teks  Tambahan       8
2     Teks     Utama       2


,id,kategori,tipe,prompt,target_evaluasi
0,1,Teks,Utama,"In the future, quantization for large language...",Prediksi masa depan quantization
1,2,Teks,Utama,Explain quantization entanglement in simple terms,Penjelasan konsep teknis
2,3,Teks,Tambahan,What are the advantages of INT8 quantization f...,Keunggulan INT8
3,4,Teks,Tambahan,How does post-training quantization differ fro...,Perbandingan PTQ vs QAT
4,5,Teks,Tambahan,Describe the challenges of deploying LLMs on e...,Tantangan edge deployment
5,6,Teks,Tambahan,What is the role of BitsAndBytes library in LL...,Library quantization
6,7,Teks,Tambahan,Explain how BLEU score measures text generatio...,Evaluasi metrik BLEU
7,8,Teks,Tambahan,What is weight quantization and why does it ma...,Konsep dasar quantization
8,9,Teks,Tambahan,How does memory bandwidth affect LLM inference...,Hardware dan inferensi
9,10,Teks,Tambahan,Compare FP16 and INT8 precision formats for ne...,Perbandingan format presisi


## 3. Konfigurasi yang siap untuk analisis


In [ ]:
# Ringkasan konfigurasi yang akan digunakan di notebook 02-05
configs = [
    ('GPT-2', 'RTX4070'),
    ('GPT-2', 'RTX4080'),
    ('LLaMA-2-7B-Chat', 'RTX4070'),
    ('LLaMA-2-7B-Chat', 'RTX4080'),
    ('Qwen1.5-1.8B-Chat', 'RTX4070'),
    ('Qwen1.5-1.8B-Chat', 'RTX4080'),
]

print('✅ Konfigurasi siap untuk workflow:')
for i, (model, gpu) in enumerate(configs, 1):
    print(f'  {i}. {model:25s} + {gpu}')


ID 1: In the future, quantization for large language models will
ID 2: Explain quantization entanglement in simple terms


## 4. Next: Jalankan Notebook 02 untuk Benchmark Simulasi


In [ ]:
print()
print('=' * 70)
print('📌 WORKFLOW ORDER:')
print('=' * 70)
print('1️⃣  Notebook 01 (ini) - Persiapan data ✓')
print('2️⃣  Notebook 02 - Benchmark hasil simulasi')
print('3️⃣  Notebook 03 - Evaluasi metrik kualitas')
print('4️⃣  Notebook 04 - Visualisasi hasil')
print('5️⃣  Notebook 05 - Penentuan model terbaik')
print()
print('Setiap notebook menggunakan data simulasi dari jurnal Oprea & Bâra (2026)')
print('untuk mendemonstrasikan hasil penelitian tanpa perlu GPU NVIDIA.')
print('=' * 70)


Dataset siap. Ringkasan:
kategori
Teks    10
Kode    10
Name: count, dtype: int64
